# Local Agentic AI with Ollama & Gemma 3 — Lab 1 Guide

This notebook demonstrates setting up and running **Google Gemma 3 (4B)** locally using **Ollama** — a high-performance inference engine optimized for Apple Silicon (Metal) and modern GPUs. This forms the foundational local LLM layer for autonomous AI agents with zero cloud subscription costs, no rate limits, and 100% data privacy.

| Model Variant | Parameters | Quantization | Local Size | VRAM / RAM | Best Use Case |
|---|---|---|---|---|---|
| `gemma3:1b` | 1.2 B | Q4_K_M | ~1.1 GB | ~2 GB | Ultra-fast edge devices, mobile, IoT |
| `gemma3:4b` | 4.3 B | Q4_K_M | ~3.3 GB | ~4.5 GB | ✅ **Recommended sweet spot** for local agents & tool calling |
| `gemma3:12b` | 12.1 B | Q4_K_M | ~7.8 GB | ~9.5 GB | High-complexity reasoning & advanced code generation |
| `gemma3:27b` | 27.2 B | Q4_K_M | ~17 GB | ~20 GB | Frontier open-weights desktop deployment |

> **Why Local LLMs for Agents?** Autonomous agents execute dozens or hundreds of prompt-and-response loops for chain-of-thought reasoning, self-reflection, and tool invocation. Running `gemma3:4b` locally eliminates per-token API fees, network latency, and third-party data transmission, providing a robust, deterministic execution environment.

## 1. Prerequisites & System Setup

### System Requirements
- **macOS** (Apple Silicon M1/M2/M3/M4 recommended for Metal acceleration), Linux, or Windows
- **Ollama CLI & Daemon**: Installed via `brew install ollama` (macOS) or `curl -fsSL https://ollama.com/install.sh | sh` (Linux)
- **Model**: `gemma3:4b` pulled to local storage via `ollama pull gemma3:4b` (~3.3 GB)
- **Python**: 3.10+ with `requests`, `openai`, and `IPython`

In [ ]:
# ── Environment Setup & Model Verification ────────────────────────────────────
# Ensure OpenAI client and dependencies are installed in the active kernel
%pip install -q openai requests ipython

!ollama --version
!ollama list

]11;?\ollama version is 0.18.2
]11;?\

## 2. Check Ollama Daemon Connectivity

Ollama runs as a background service listening on `http://localhost:11434`. Before initiating agent tasks, verify that the local HTTP server is active and responsive.

In [ ]:
import requests

OLLAMA_BASE_URL = "http://localhost:11434"

try:
    resp = requests.get(OLLAMA_BASE_URL, timeout=5)
    if resp.status_code == 200:
        print(f"✅ Ollama server is running: {resp.text.strip()}")
    else:
        print(f"⚠️ Server returned status code: {resp.status_code}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Ollama server is running: Ollama is running


## 3. Native REST API & Token Streaming

Ollama exposes a native `/api/generate` endpoint. Streaming token responses line-by-line enables real-time user interfaces and low-latency agent reasoning loops.

In [ ]:
import json
import requests

MODEL_NAME = "gemma3:4b"

payload = {
    "model": MODEL_NAME,
    "prompt": "Write a short Python function that reverses a string.",
    "stream": True
}

print(f"Sending prompt to {MODEL_NAME}...\n")
resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, stream=True)

for line in resp.iter_lines():
    if not line:
        continue
    chunk = json.loads(line)
    token = chunk.get("response", "")
    print(token, end="", flush=True)

Sending prompt to gemma3:4b...

```python
def reverse_string(s):
  """Reverses a string.

  Args:
    s: The string to reverse.

  Returns:
    The reversed string.
  """
  return s[::-1]

# Example usage:
string = "hello"
reversed_string = reverse_string(string)
print(reversed_string)  # Output: olleh
```

**Explanation:**
1. **`def reverse_string(s):`**: Defines a function named `reverse_string` taking argument `s`.
2. **`return s[::-1]`**: Uses Python's extended slicing with a step of -1 to reverse the string in O(n) time.


## 4. OpenAI-Compatible SDK Integration

Ollama provides an OpenAI-compatible `/v1` endpoint. This allows you to use standard `openai` client libraries and agent frameworks (LangChain, AutoGen, CrewAI) without rewriting integration logic.

In [ ]:
try:
    from openai import OpenAI
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    from openai import OpenAI

# Connect to Ollama's local OpenAI-compatible endpoint
client = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")

print("✓ OpenAI client initialized.")
print(f"  Base URL : {client.base_url}")
print(f"  Target   : {MODEL_NAME}")

✓ OpenAI client initialized.
  Base URL : http://localhost:11434/v1/
  Target   : gemma3:4b


## 5. Structured Chat Completion Request

Now we query the model using structured chat messages (`system` prompt and `user` query). We request a Markdown comparison table to test structured reasoning and generation capabilities.

In [ ]:
prompt_text = "Compare the population of New York and Los Angeles. Provide the answer in a markdown table format."

messages = [
    {"role": "system", "content": "You are a helpful and precise technical assistant specialized in structured data."},
    {"role": "user", "content": prompt_text}
]

print(f"User Request:\n{prompt_text}\n")
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    temperature=0.2
)

answers = response.choices[0].message.content
print("✓ Response received from Gemma 3.")

User Request:
Compare the population of New York and Los Angeles. Provide the answer in a markdown table format.

✓ Response received from Gemma 3.


## 6. Raw Chat Response Inspection

Inspect the raw text content returned by `gemma3:4b` prior to rendering.

In [ ]:
print("Raw Model Output:\n" + "=" * 50)
print(answers)

Raw Model Output:
Here is a comparison of the population of New York and Los Angeles, presented in a markdown table format:

| City | Population (Estimate - 2023) | Metropolitan Area Population (Estimate - 2023) |
|---|---|---|
| **New York City** | 8,804,190 | 20,274,420 |
| **Los Angeles** | 3,898,747 | 12,897,256 |

**Source:** U.S. Census Bureau QuickFacts


## 7. Render Rich Markdown Output

In interactive notebooks, we render the model's Markdown response into a styled, publication-ready table using `IPython.display.Markdown`.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(answers))

Here is a comparison of the population of New York and Los Angeles, presented in a markdown table format:

| City | Population (Estimate - 2023) | Metropolitan Area Population (Estimate - 2023) |
|---|---|---|
| **New York City** | 8,804,190 | 20,274,420 |
| **Los Angeles** | 3,898,747 | 12,897,256 |

**Source:** U.S. Census Bureau QuickFacts
